# Soccerpoly | Coach hint and Continental Cup

**Product question.** Does a one-line hint move solo entrants to a deliberate first route within 90 seconds? Completion must stay within a −3 pp margin. The onboarding records are synthetic. Economy rows come from batches executed with the actual Luau rules.


In [ ]:
from pathlib import Path
import json
import sqlite3

candidates = [Path.cwd() / "dist/analytics", Path.cwd().parent / "dist/analytics"]
report_dir = next(path for path in candidates if (path / "experiment.json").is_file())
result = json.loads((report_dir / "experiment.json").read_text())
lineage = json.loads((report_dir / "lineage.json").read_text())
print(result["dataset"], lineage["accepted"], "quality-checked events")
print(lineage["marts"])


## Intention to treat

Deduplicate by event ID. Keep the first assignment per player, including players who never choose a route. Diagnose allocation before reading effect sizes. Report the device and acquisition-channel cuts as exploratory.


In [ ]:
with sqlite3.connect(report_dir / "worlddraft.sqlite") as conn:
    overview = conn.execute('''
       SELECT dimension, segment, variant, players,
              ROUND(100.0 * routes / players, 1) AS route_90s_pct,
              ROUND(100.0 * finishing / players, 1) AS finish_pct,
              ROUND(100.0 * d1_returned / NULLIF(d1_eligible, 0), 1) AS d1_pct
       FROM segment_metrics ORDER BY dimension, segment, variant
    ''').fetchall()
for row in overview:
    print(row)


In [ ]:
primary = result["first_route_within_90s"]
guardrail = result["finish_guardrail"]
for name, metric in [("First route", primary), ("Completion", guardrail)]:
    print(f"{name}: {metric['delta']*100:+.2f} pp, "
          f"[{metric['ci95_lower']*100:+.2f}, {metric['ci95_upper']*100:+.2f}]")
print("Sample ratio mismatch p:", result["srm_p"])
print("Decision:", result["decision"], "| completion plan per arm:", result["guardrail_minimum_per_arm"])
assert result["dataset"] == "synthetic_ab" or result["dataset"] == "playtest"


## Table-size economy

How does a full table change club availability and transfer fees? Use the actual rule engine to screen for failure modes. Compare statistics per seat and per table with their own denominators.


In [ ]:
with sqlite3.connect(report_dir / "worlddraft.sqlite") as conn:
    for row in conn.execute('''
        SELECT seats, tables, avg_ending_cash, avg_clubs, avg_fees, goals_per_cup_visit
        FROM economy ORDER BY seats
    '''):
        print(row)


## Decision

The fixture has an apparent activation improvement. Completion does not establish the planned non-inferiority margin. Keep the variant eligible for the closed playtest, inspect first-route timing by device, and check whether larger tables leave too few clubs available before changing prices. D1 uses a rolling 24–48h window with censoring. Marketing source labels in the fixture are modeled.
